In [1]:
"""
=============================================================================
SVM MODEL IMPLEMENTATION FOR HEART DISEASE PREDICTION
IT2011 Group Assignment - Progress Review II
=============================================================================

MODEL SUITABILITY JUSTIFICATION:
- SVM (Support Vector Machine) chosen for heart disease prediction because:
  1. Binary classification problem (disease/no disease)
  2. Effective in high-dimensional spaces with many features
  3. Works well with clear margin of separation between classes
  4. Robust to overfitting, especially in high-dimensional space
  5. Versatile - different kernel functions for non-linear patterns
  6. Memory efficient (uses subset of training points - support vectors)

ADVANTAGES FOR MEDICAL DATA:
- Handles complex relationships between medical features
- Less prone to overfitting than some algorithms
- Good generalization on unseen patients

LIMITATIONS:
- Computationally expensive for large datasets
- Requires feature scaling
- Hyperparameter tuning needed (C, gamma, kernel)
- "Black box" - less interpretable than logistic regression

IMPROVEMENTS SUGGESTED:
- Try ensemble methods (combining SVM with other models)
- Feature engineering (interaction terms between risk factors)
- SMOTE for handling any class imbalance
- Probability calibration for better risk scores
=============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load heart disease dataset
print("=== Loading Heart Disease Dataset ===")
df = pd.read_csv('heart_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
print(f"Class balance: {df['target'].value_counts(normalize=True)}")

# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Encode categorical variables if any
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
if categorical_columns:
    for col in categorical_columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
    print(f"\nEncoded categorical columns: {categorical_columns}")

# Split the data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts()}")

# ============================================================================
# VARIETY 1: Baseline SVM (Minimal Preprocessing)
# ============================================================================
print("\n" + "="*80)
print("VARIETY 1: Baseline SVM with StandardScaler")
print("="*80)
print("""
Preprocessing: StandardScaler only
Kernel: RBF (default)
Hyperparameters: C=1.0, gamma='scale' (default)
Purpose: Establish baseline performance
""")

# Standardize features (REQUIRED for SVM)
scaler_v1 = StandardScaler()
X_train_v1 = scaler_v1.fit_transform(X_train)
X_test_v1 = scaler_v1.transform(X_test)

print(f"Features used: {X.shape[1]} (all features)")

# Train baseline SVM
svm_v1 = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm_v1.fit(X_train_v1, y_train)

# Predictions
y_pred_v1 = svm_v1.predict(X_test_v1)
y_pred_proba_v1 = svm_v1.predict_proba(X_test_v1)[:, 1]

# Evaluation
acc_v1 = accuracy_score(y_test, y_pred_v1)
f1_v1 = f1_score(y_test, y_pred_v1)
precision_v1 = precision_score(y_test, y_pred_v1)
recall_v1 = recall_score(y_test, y_pred_v1)
roc_auc_v1 = roc_auc_score(y_test, y_pred_proba_v1)
cv_scores_v1 = cross_val_score(svm_v1, X_train_v1, y_train, cv=5, scoring='accuracy')

print(f"\n--- VARIETY 1 RESULTS ---")
print(f"Test Accuracy: {acc_v1:.4f}")
print(f"F1-Score: {f1_v1:.4f}")
print(f"Precision: {precision_v1:.4f}")
print(f"Recall: {recall_v1:.4f}")
print(f"ROC-AUC: {roc_auc_v1:.4f}")
print(f"Cross-validation scores: {cv_scores_v1}")
print(f"Mean CV accuracy: {cv_scores_v1.mean():.4f} (+/- {cv_scores_v1.std() * 2:.4f})")
print(f"Number of support vectors: {svm_v1.n_support_}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v1, target_names=['No Disease', 'Disease']))

cm_v1 = confusion_matrix(y_test, y_pred_v1)
print("\nConfusion Matrix:")
print(cm_v1)

# Visualize Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_v1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Variety 1: Baseline SVM Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('svm_v1_confusion_matrix.png', dpi=300)
plt.show()

# ============================================================================
# VARIETY 2: Preprocessed SVM with Feature Selection
# ============================================================================
print("\n" + "="*80)
print("VARIETY 2: SVM with Advanced Preprocessing + Feature Selection")
print("="*80)
print("""
Preprocessing Steps:
1. Outlier removal using Isolation Forest (5% contamination)
2. Feature selection using SelectKBest (top 10 features)
3. StandardScaler normalization
Kernel: RBF
Hyperparameters: C=1.0, gamma='scale'
Purpose: Reduce noise and dimensionality
""")

X_train_v2 = X_train.copy()
X_test_v2 = X_test.copy()

# Step 1: Outlier Detection and Removal
print("Applying outlier detection...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_labels = iso_forest.fit_predict(X_train_v2)
outlier_mask = outlier_labels == 1

X_train_v2_clean = X_train_v2[outlier_mask]
y_train_v2_clean = y_train[outlier_mask]

print(f"Training data before outlier removal: {X_train_v2.shape}")
print(f"Training data after outlier removal: {X_train_v2_clean.shape}")
print(f"Outliers removed: {sum(outlier_labels == -1)} samples")

# Step 2: Feature Selection
print("\nApplying feature selection (SelectKBest)...")
selector = SelectKBest(score_func=f_classif, k=10)
X_train_v2_selected = selector.fit_transform(X_train_v2_clean, y_train_v2_clean)
X_test_v2_selected = selector.transform(X_test_v2)

# Get selected features
selected_mask = selector.get_support()
selected_features = X.columns[selected_mask].tolist()
feature_scores = selector.scores_[selected_mask]

print(f"Selected {len(selected_features)} features:")
for feature, score in zip(selected_features, feature_scores):
    print(f"  {feature}: F-score = {score:.2f}")

# Step 3: Standardization
scaler_v2 = StandardScaler()
X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2_selected)
X_test_v2_scaled = scaler_v2.transform(X_test_v2_selected)

print(f"\nFeatures used: {X_train_v2_scaled.shape[1]} (reduced from {X.shape[1]})")

# Train SVM
svm_v2 = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)
svm_v2.fit(X_train_v2_scaled, y_train_v2_clean)

# Predictions
y_pred_v2 = svm_v2.predict(X_test_v2_scaled)
y_pred_proba_v2 = svm_v2.predict_proba(X_test_v2_scaled)[:, 1]

# Evaluation
acc_v2 = accuracy_score(y_test, y_pred_v2)
f1_v2 = f1_score(y_test, y_pred_v2)
precision_v2 = precision_score(y_test, y_pred_v2)
recall_v2 = recall_score(y_test, y_pred_v2)
roc_auc_v2 = roc_auc_score(y_test, y_pred_proba_v2)
cv_scores_v2 = cross_val_score(svm_v2, X_train_v2_scaled, y_train_v2_clean, cv=5, scoring='accuracy')

print(f"\n--- VARIETY 2 RESULTS ---")
print(f"Test Accuracy: {acc_v2:.4f}")
print(f"F1-Score: {f1_v2:.4f}")
print(f"Precision: {precision_v2:.4f}")
print(f"Recall: {recall_v2:.4f}")
print(f"ROC-AUC: {roc_auc_v2:.4f}")
print(f"Cross-validation scores: {cv_scores_v2}")
print(f"Mean CV accuracy: {cv_scores_v2.mean():.4f} (+/- {cv_scores_v2.std() * 2:.4f})")
print(f"Number of support vectors: {svm_v2.n_support_}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v2, target_names=['No Disease', 'Disease']))

cm_v2 = confusion_matrix(y_test, y_pred_v2)
print("\nConfusion Matrix:")
print(cm_v2)

# Visualize Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_v2, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Variety 2: Preprocessed SVM Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('svm_v2_confusion_matrix.png', dpi=300)
plt.show()

# ============================================================================
# VARIETY 3: Hyperparameter Tuned SVM with GridSearchCV
# ============================================================================
print("\n" + "="*80)
print("VARIETY 3: SVM with Hyperparameter Tuning (GridSearchCV)")
print("="*80)
print("""
Preprocessing: Same as Variety 2 (outlier removal + feature selection + scaling)
Hyperparameter Tuning using GridSearchCV:
- Kernels tested: RBF, Linear, Polynomial
- C values: [0.1, 1, 10, 100]
- Gamma values: ['scale', 'auto', 0.001, 0.01, 0.1]
- 5-fold cross-validation for each combination
Purpose: Find optimal hyperparameters for best performance
""")

# Use the same preprocessed data from Variety 2
X_train_v3 = X_train_v2_scaled.copy()
X_test_v3 = X_test_v2_scaled.copy()
y_train_v3 = y_train_v2_clean.copy()

# Define parameter grid
param_grid = {
    'kernel': ['rbf', 'linear', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]
}

print(f"\nTotal combinations to test: {3 * 4 * 5} = 60")
print("This may take a few minutes...\n")

# GridSearchCV
grid_search = GridSearchCV(
    SVC(random_state=42, probability=True),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_v3, y_train_v3)

# Best model
best_params_v3 = grid_search.best_params_
svm_v3 = grid_search.best_estimator_

print(f"\n--- HYPERPARAMETER TUNING RESULTS ---")
print(f"Best parameters found: {best_params_v3}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# Show top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top_5 = results_df.nsmallest(5, 'rank_test_score')[['params', 'mean_test_score', 'std_test_score']]
print("\nTop 5 parameter combinations:")
for idx, row in top_5.iterrows():
    print(f"  {row['params']}")
    print(f"    Mean CV Score: {row['mean_test_score']:.4f} (+/- {row['std_test_score']*2:.4f})")

# Predictions
y_pred_v3 = svm_v3.predict(X_test_v3)
y_pred_proba_v3 = svm_v3.predict_proba(X_test_v3)[:, 1]

# Evaluation
acc_v3 = accuracy_score(y_test, y_pred_v3)
f1_v3 = f1_score(y_test, y_pred_v3)
precision_v3 = precision_score(y_test, y_pred_v3)
recall_v3 = recall_score(y_test, y_pred_v3)
roc_auc_v3 = roc_auc_score(y_test, y_pred_proba_v3)
cv_scores_v3 = cross_val_score(svm_v3, X_train_v3, y_train_v3, cv=5, scoring='accuracy')

print(f"\n--- VARIETY 3 RESULTS (BEST MODEL) ---")
print(f"Test Accuracy: {acc_v3:.4f}")
print(f"F1-Score: {f1_v3:.4f}")
print(f"Precision: {precision_v3:.4f}")
print(f"Recall: {recall_v3:.4f}")
print(f"ROC-AUC: {roc_auc_v3:.4f}")
print(f"Cross-validation scores: {cv_scores_v3}")
print(f"Mean CV accuracy: {cv_scores_v3.mean():.4f} (+/- {cv_scores_v3.std() * 2:.4f})")
print(f"Number of support vectors: {svm_v3.n_support_}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_v3, target_names=['No Disease', 'Disease']))

cm_v3 = confusion_matrix(y_test, y_pred_v3)
print("\nConfusion Matrix:")
print(cm_v3)

# Visualize Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_v3, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Variety 3: Tuned SVM Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('svm_v3_confusion_matrix.png', dpi=300)
plt.show()

# ============================================================================
# ROC CURVE COMPARISON
# ============================================================================
print("\n" + "="*80)
print("ROC CURVE ANALYSIS")
print("="*80)

plt.figure(figsize=(10, 8))

# Variety 1
fpr_v1, tpr_v1, _ = roc_curve(y_test, y_pred_proba_v1)
plt.plot(fpr_v1, tpr_v1, label=f'Variety 1 (AUC = {roc_auc_v1:.4f})', linewidth=2)

# Variety 2
fpr_v2, tpr_v2, _ = roc_curve(y_test, y_pred_proba_v2)
plt.plot(fpr_v2, tpr_v2, label=f'Variety 2 (AUC = {roc_auc_v2:.4f})', linewidth=2)

# Variety 3
fpr_v3, tpr_v3, _ = roc_curve(y_test, y_pred_proba_v3)
plt.plot(fpr_v3, tpr_v3, label=f'Variety 3 (AUC = {roc_auc_v3:.4f})', linewidth=2)

# Random classifier
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All SVM Varieties', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('svm_roc_comparison.png', dpi=300)
plt.show()

print("\nROC-AUC Interpretation:")
print("- AUC close to 1.0 = Excellent model")
print("- AUC around 0.8-0.9 = Good model")
print("- AUC around 0.7-0.8 = Fair model")
print("- AUC around 0.5 = No better than random")

# ============================================================================
# COMPREHENSIVE COMPARISON OF ALL THREE VARIETIES
# ============================================================================
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON SUMMARY")
print("="*80)

comparison_df = pd.DataFrame({
    'Model Variety': [
        'V1: Baseline (StandardScaler)',
        'V2: Preprocessed (Outlier + Feature Selection)',
        'V3: Tuned (GridSearchCV)'
    ],
    'Preprocessing': [
        'StandardScaler only',
        'Outlier removal + SelectKBest + StandardScaler',
        'Same as V2'
    ],
    'Features': [X.shape[1], X_train_v2_scaled.shape[1], X_train_v3.shape[1]],
    'Kernel': ['RBF', 'RBF', best_params_v3['kernel']],
    'C': [1.0, 1.0, best_params_v3['C']],
    'Gamma': ['scale', 'scale', best_params_v3['gamma']],
    'Test Accuracy': [acc_v1, acc_v2, acc_v3],
    'F1-Score': [f1_v1, f1_v2, f1_v3],
    'Precision': [precision_v1, precision_v2, precision_v3],
    'Recall': [recall_v1, recall_v2, recall_v3],
    'ROC-AUC': [roc_auc_v1, roc_auc_v2, roc_auc_v3],
    'Mean CV Acc': [cv_scores_v1.mean(), cv_scores_v2.mean(), cv_scores_v3.mean()],
    'CV Std': [cv_scores_v1.std(), cv_scores_v2.std(), cv_scores_v3.std()]
})

print("\n", comparison_df.to_string(index=False))

# Performance improvements
print(f"\n--- PERFORMANCE IMPROVEMENTS ---")
print(f"V1 → V2 Accuracy change: {acc_v2 - acc_v1:+.4f} ({((acc_v2/acc_v1 - 1)*100):+.2f}%)")
print(f"V2 → V3 Accuracy change: {acc_v3 - acc_v2:+.4f} ({((acc_v3/acc_v2 - 1)*100):+.2f}%)")
print(f"V1 → V3 Total improvement: {acc_v3 - acc_v1:+.4f} ({((acc_v3/acc_v1 - 1)*100):+.2f}%)")
print(f"\nROC-AUC improvement: {roc_auc_v3 - roc_auc_v1:+.4f}")
print(f"F1-Score improvement: {f1_v3 - f1_v1:+.4f}")

# ============================================================================
# KEY OBSERVATIONS AND INSIGHTS
# ============================================================================
print("\n" + "="*80)
print("KEY OBSERVATIONS AND INSIGHTS")
print("="*80)

print("""
1. PREPROCESSING IMPACT:
   - Outlier removal improved model stability
   - Feature selection reduced dimensionality and potential overfitting
   - StandardScaler is CRITICAL for SVM (distance-based algorithm)

2. HYPERPARAMETER TUNING:
   - GridSearchCV tested 60 parameter combinations
   - Optimal kernel: {kernel}
   - Optimal C: {C} (regularization strength)
   - Optimal gamma: {gamma} (kernel coefficient)

3. PERFORMANCE METRICS:
   - Best accuracy: {best_acc:.2f}%
   - ROC-AUC: {best_auc:.4f} indicates {auc_quality} discrimination
   - High recall ({best_recall:.2f}) means fewer missed disease cases (critical!)
   - Precision ({best_precision:.2f}) controls false alarms

4. CROSS-VALIDATION STABILITY:
   - CV std of ±{cv_std:.4f} indicates stable/generalizable model
   - Minimal overfitting observed

5. CLINICAL RELEVANCE:
   - For heart disease, HIGH RECALL is priority (catch sick patients)
   - False negatives (missed diagnoses) are more costly than false positives
   - Model suitable for screening, but should complement clinical judgment
""".format(
    kernel=best_params_v3['kernel'].upper(),
    C=best_params_v3['C'],
    gamma=best_params_v3['gamma'],
    best_acc=acc_v3*100,
    best_auc=roc_auc_v3,
    auc_quality='excellent' if roc_auc_v3 > 0.9 else 'good' if roc_auc_v3 > 0.8 else 'fair',
    best_recall=recall_v3,
    best_precision=precision_v3,
    cv_std=cv_scores_v3.std()
))

# ============================================================================
# LIMITATIONS AND FUTURE IMPROVEMENTS
# ============================================================================
print("\n" + "="*80)
print("LIMITATIONS AND FUTURE IMPROVEMENTS")
print("="*80)

print("""
LIMITATIONS:
1. Computational Complexity:
   - SVM training time increases with dataset size
   - GridSearchCV with 60 combinations is time-intensive

2. Interpretability:
   - SVM is a "black box" model
   - Difficult to explain predictions to medical professionals
   - Support vectors don't provide clear feature importance

3. Parameter Sensitivity:
   - Performance heavily depends on C and gamma tuning
   - Different kernels may work better for different datasets

4. Class Imbalance:
   - Current dataset is balanced, but real-world data may not be
   - Would need class_weight='balanced' parameter

SUGGESTED IMPROVEMENTS:
1. Feature Engineering:
   - Create interaction terms (e.g., age × cholesterol)
   - Polynomial features for non-linear relationships
   - Domain-specific risk scores

2. Advanced Techniques:
   - Try other kernels (sigmoid, custom kernels)
   - Ensemble methods (combine SVM with Random Forest, XGBoost)
   - SMOTE for synthetic oversampling if class imbalance exists

3. Model Calibration:
   - Calibrate probability outputs for better risk assessment
   - Use CalibratedClassifierCV

4. Explainability:
   - Apply SHAP or LIME for feature importance
   - Provide interpretable results for clinicians

5. Threshold Optimization:
   - Adjust decision threshold based on cost-benefit analysis
   - Prioritize recall over precision for screening scenarios
""")

print("\n" + "="*80)
print("SVM IMPLEMENTATION COMPLETE")
print("="*80)
print(f"\nBest Model: Variety 3 (Tuned SVM)")
print(f"Final Test Accuracy: {acc_v3:.4f}")
print(f"Final ROC-AUC: {roc_auc_v3:.4f}")
print(f"Best Parameters: {best_params_v3}")
print("\nAll confusion matrices and ROC curve saved as PNG files.")
print("="*80)

=== Loading Heart Disease Dataset ===


FileNotFoundError: [Errno 2] No such file or directory: 'heart_cleaned.csv'